In [1]:
from langchain_aws import ChatBedrockConverse
import os
from dotenv import load_dotenv

load_dotenv()
model = ChatBedrockConverse(
    model=os.getenv("AWS_MODEL_ID"),
    region_name=os.getenv("AWS_REGION"),)

In [2]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dataclass
class LanguageContext:
    user_language: str = "English"

@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_language = request.runtime.context.user_language
    base_prompt = "You are a helpful assistant."

    if user_language != "English":
        return f"{base_prompt} only respond in {user_language}."
    elif user_language == "English":
        return base_prompt

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    context_schema=LanguageContext,
    middleware=[user_language_prompt]
)

In [4]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Irish")
)

print(response["messages"][-1].content)

Hello! Go raibh maith agat, conas atá tú?


In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Spanish")
)

print(response["messages"][-1].content)

¡Hola! Estoy aquí para ayudarte. Como asistente virtual, no tengo sentimientos, pero estoy listo y dispuesto a asistirte en lo que necesites. ¿En qué puedo ayudarte hoy?


In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="French")
)

print(response["messages"][-1].content)

Bonjour ! Je vais bien, merci de demander. Et toi, comment vas-tu ? 

Je suis là pour t’aider avec toute question ou sujet dont tu voudrais discuter. Que puis-je faire pour toi aujourd’hui ?
